## Hybrid Retriever: Combining Dense and Sparse Retriever

A **Hybrid Retriever** combines the strengths of **Dense Retrieval** (Semantic/Vector Search using Embeddings) and **Sparse Retrieval** (Keyword/Lexical Search using BM25) to achieve higher precision and recall in RAG pipelines.

### 💡 When to Use BM25 + Semantic Search Together?

| **Use Case** | **Why Hybrid Retrieval Helps** |
| :--- | :--- |
| **RAG Pipelines** | Prevents retrieval hallucination by ensuring both exact keyword matches and fuzzy semantic context are considered. |
| **Technical Documentation Search** | Developers may search *"how to use API"* while the doc mentions *"API usage"* — combining BM25 and semantic search significantly improves hit rate. |
| **Legal & Medical QA** | Queries often require exact statutory/drug term matching (BM25), while also requiring deep conceptual and contextual understanding (Dense). |
| **E-commerce & Product Search** | *"cheap noise-canceling headphones"* matches *"affordable ANC earbuds"* via dense search, while BM25 confirms exact model names and acronyms like *"ANC"*. |
| **Multilingual / Cross-lingual Retrieval** | Semantic models bridge language and vocabulary differences, while BM25 ensures exact matches when terms are in the same language. |
| **Customer Support Chatbots / FAQs** | Real users often type vague or keyword-heavy queries — hybrid retrieval improves answer reliability and intent matching. |
| **Transcripts / Unstructured Data** | Speech-to-text transcripts or raw emails contain inconsistent phrasing — dense retrieval captures semantic intent, while sparse confirms key terms. |

### 🚀 Key Benefits of Combining BM25 + Semantic Search

| # | **Benefit** | **Explanation** |
| :-: | :--- | :--- |
| **1** | **Boosts Recall** | BM25 catches exact keyword matches that dense retrieval misses; dense captures meaning even if keywords differ. Together, you minimize the risk of missing relevant documents. |
| **2** | **Handles Synonyms & Rephrasing** | Semantic search matches *"create app"* with *"build LLM system"* even if there are no shared words, while BM25 locks onto exact terms like *"LLM"* and *"app"*. |
| **3** | **Improves Retrieval Robustness** | Covers both users who search by specific terms (*"LangChain agent"*) and those who use natural phrasing (*"how do I use LangChain to talk to tools"*). |
| **4** | **Supports Lexical Importance** | BM25 scores rare keywords higher — this is crucial in technical/legal/medical contexts where a rare term (like *"osteoporosis"*) should weigh heavily. |
| **5** | **Bridges Document Diversity** | In large corpora (e.g., web pages, PDFs, blogs), you often have a mix of well-structured and loosely written text. Hybrid retrieval adapts to both. |
| **6** | **Easy to Tune via Weights** | You can easily adjust the influence of each method (e.g., `weights=[0.7, 0.3]` for 70% Dense + 30% Sparse). |
| **7** | **Helps with Misspellings or Variants** | Dense models are more tolerant to typos and misspelled words (e.g., *"11m aplication"*), compensating where BM25 exact match may fail. |

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.schema import Document


In [ ]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

# Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

In [ ]:
### Sparse Retriever(BM25)
sparse_retriever=BM25Retriever.from_documents(docs)
sparse_retriever.k=3 ##top- k documents to retriever

## step 4 : Combine with Ensemble Retriever
hybrid_retriever=EnsembleRetriever(
    retrievers=[dense_retriever,sparse_retriever],
    weight=[0.7,0.3]
)


In [ ]:
hybrid_retriever

In [ ]:
# Step 5: Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")

### RAG Pipeline with hybrid retriever

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

In [ ]:
# Step 5: Prompt Template
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

## step 6-llm
llm=init_chat_model("openai:gpt-3.5-turbo",temperature=0.2)
llm

In [ ]:
### Create stuff Docuemnt Chain
document_chain=create_stuff_documents_chain(llm=llm,prompt=prompt)

## create Full rAg chain
rag_chain=create_retrieval_chain(retriever=hybrid_retriever,combine_docs_chain=document_chain)
rag_chain


In [ ]:
# Step 9: Ask a question
query = {"input": "How can I build an app using LLMs?"}
response = rag_chain.invoke(query)

# Step 10: Output
print("✅ Answer:\n", response["answer"])

print("\n📄 Source Documents:")
for i, doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")